# AVD894 — UVFA Treasure Hunt

This notebook runs the implementation through Intermediate Task 2. It is an orchestration and presentation notebook: the maintained implementation remains in `src/` and `scripts/`.

Paper: Tom Schaul et al., *Universal Value Function Approximators*, ICML 2015.

The simulator is a course-project adaptation. Results are generated locally and are not copied from the paper.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess
import time

# This works whether the notebook is opened from the project root or notebooks/.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook inside New Paper Project')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

CONFIG_TASK1 = PROJECT_ROOT / 'configs' / 'task1_small.json'
CONFIG_TASK2 = PROJECT_ROOT / 'configs' / 'task2_default.json'
ARTIFACTS = PROJECT_ROOT / 'artifacts' / 'notebook_run'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_ROOT)
print('Artifacts:', ARTIFACTS)

## 1. Runtime check

The project automatically uses CUDA when PyTorch reports it as available and otherwise falls back to CPU.

In [ ]:
import numpy as np
import torch

print('Python executable:', sys.executable)
print('NumPy:', np.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Intermediate Task 1 — exact value iteration

This cell solves the finite state-goal MDP and stores the same JSON/CSV artifacts as the command-line script.

In [ ]:
from goal_treasure.exact import follow_policy, value_iteration
from goal_treasure.utils import load_json, make_env, save_json

task1_config = load_json(CONFIG_TASK1)
task1_env = make_env(task1_config)
task1_result = value_iteration(
    task1_env.obstacles,
    gamma=task1_config['gamma'],
    step_reward=task1_config['step_reward'],
    wall_penalty=task1_config['wall_penalty'],
    goal_reward=task1_config['goal_reward'],
    tolerance=task1_config['tolerance'],
    max_iterations=task1_config['max_iterations'],
)
start, goal = task1_env.free_cells[0], task1_env.free_cells[-1]
task1_path = follow_policy(task1_result.policy, start, goal, task1_env.obstacles)
task1_summary = {
    'state_goal_pairs': len(task1_result.values),
    'free_cells': len(task1_env.free_cells),
    'actions': task1_env.action_size,
    'iterations': task1_result.iterations,
    'final_delta': task1_result.delta,
    'example_path_length': len(task1_path) - 1,
}
save_json(ARTIFACTS / 'task1_summary.json', task1_summary)
print(json.dumps(task1_summary, indent=2))
print('Path:', task1_path)

In [ ]:
# Simple notebook visualization for Task 1's exact path.
from IPython.display import display, Markdown

task1_env.reset(start=start, goal=goal)
display(Markdown('```text\n' + task1_env.render_ascii(task1_path) + '\n```'))

## 3. Intermediate Task 2 — train from this notebook

Change `TASK2_EPISODES` when you want a longer experiment. The default is deliberately short so that the notebook remains safe to run during development.

In [ ]:
TASK2_EPISODES = 20
TASK2_OUTPUT = ARTIFACTS / f'training_{TASK2_EPISODES}_episodes'

train_command = [
    sys.executable, str(PROJECT_ROOT / 'scripts' / 'train_task2.py'),
    '--config', str(CONFIG_TASK2),
    '--episodes', str(TASK2_EPISODES),
    '--output', str(TASK2_OUTPUT),
    '--device', 'auto',
]
started = time.perf_counter()
train_run = subprocess.run(train_command, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(train_run.stdout)
if train_run.returncode != 0:
    print(train_run.stderr)
    raise RuntimeError(f'Training failed with exit code {train_run.returncode}')
print(f'Training wall time: {time.perf_counter() - started:.1f} seconds')
CHECKPOINT = TASK2_OUTPUT / 'checkpoint.pt'
print('Checkpoint:', CHECKPOINT)

## 4. Evaluate on new random test episodes

The evaluation seeds are separate from the training seeds. The shortest-path controller is a reference baseline, not the RL agent.

In [ ]:
EVAL_EPISODES = 20
EVALUATION = TASK2_OUTPUT / 'evaluation.json'
eval_command = [
    sys.executable, str(PROJECT_ROOT / 'scripts' / 'evaluate_task2.py'),
    '--config', str(CONFIG_TASK2),
    '--checkpoint', str(CHECKPOINT),
    '--episodes', str(EVAL_EPISODES),
    '--output', str(EVALUATION),
    '--device', 'auto',
]
eval_run = subprocess.run(eval_command, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(eval_run.stdout)
if eval_run.returncode != 0:
    print(eval_run.stderr)
    raise RuntimeError(f'Evaluation failed with exit code {eval_run.returncode}')
evaluation = json.loads(EVALUATION.read_text(encoding='utf-8'))
display(evaluation['learned'])
display(evaluation['shortest_path_reference'])
print('Evaluation file:', EVALUATION)

## 5. Display one learned rollout

This cell displays a new random episode as text. For the full clickable window, run `scripts/demo.py` once from the terminal or VS Code task.

In [ ]:
from goal_treasure.uvfa import load_checkpoint
from goal_treasure.utils import device_from_request, make_env

demo_config = load_json(CONFIG_TASK2)
demo_env = make_env(demo_config)
device = device_from_request('auto')
network, metadata = load_checkpoint(CHECKPOINT, device)
demo_env.reset(seed=123456)
path = [demo_env.position]
total_reward = 0.0
for _ in range(demo_env.max_steps):
    with torch.no_grad():
        q_values = network(
            torch.from_numpy(demo_env.state_vector()).unsqueeze(0).to(device),
            torch.from_numpy(demo_env.goal_vector()).unsqueeze(0).to(device),
        )
    action = int(q_values.argmax(dim=1).item())
    result = demo_env.step(action)
    path.append(demo_env.position)
    total_reward += result.reward
    if result.terminated or result.truncated:
        break
print({'success': demo_env.position == demo_env.goal, 'steps': demo_env.steps, 'return': total_reward})
display(Markdown('```text\n' + demo_env.render_ascii(path) + '\n```'))

## 6. What to submit

Use the generated JSON files as evidence, and use `docs/TASK1_REPORT.md`, `docs/TASK2_REPORT.md`, and `docs/RESULTS.md` when preparing the Moodle report. Do not call the smoke-run metrics paper results. The notebook output can be exported to HTML or PDF from Jupyter for presentation.